In [ ]:
%%html
<!-- fuck vscode -->
<style>
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
</style>

In [ ]:
from typing import NamedTuple, Iterable, Self
import math
import random

import ipywidgets as w
from ipycanvas import Canvas
from palette import pick_color

In [ ]:
class XY(NamedTuple):
    x: float
    y: float

    @classmethod
    def mid(cls, p1: Self, p2: Self) -> Self:
        return cls((p1.x + p2.x) / 2, (p1.y + p2.y) / 2)

    @classmethod
    def dist(cls, p1: Self, p2: Self) -> float:
        v = XY(p2.x - p1.x, p2.y - p1.y)
        return math.sqrt(v.x * v.x + v.y * v.y)


XYs = tuple[XY, ...]

In [ ]:
def split_line(p1: XY, p2: XY, n: int) -> Iterable[XY]:
    """Split line into n segments"""
    dx = (p2.x - p1.x) / n
    dy = (p2.y - p1.y) / n
    yield p1
    for i in range(1, n):
        yield XY(p1.x + dx * i, p1.y + dy * i)
    yield p2

In [ ]:
def jig_line(points: XYs, maxoffset: float) -> Iterable[XY]:
    """Shift internal points to ±offset perpendicular to main line"""

    lng = XY.dist(points[0], points[-1])
    tng = XY((points[-1].x - points[0].x) / lng, (points[-1].y - points[0].y) / lng)
    nrm = XY(-tng.y, tng.x)

    yield points[0]
    for p in points[1:-1]:
        offset = random.uniform(-maxoffset, +maxoffset)
        yield XY(p.x + nrm.x * offset, p.y + nrm.y * offset)
    yield points[-1]

In [ ]:
def stroke_straight_path(canvas: Canvas, points: XYs):
    canvas.begin_path()
    canvas.move_to(points[0].x, points[0].y)
    for p in points[1:]:
        canvas.line_to(p.x, p.y)
    canvas.stroke()

In [ ]:
def stroke_quadsmooth_path(canvas: Canvas, points: XYs):
    """Stroking through midpoints using original points as controls"""

    midpoints = [XY.mid(points[i], points[i + 1]) for i in range(len(points) - 1)]
    canvas.begin_path()
    canvas.move_to(points[0].x, points[0].y)
    canvas.line_to(midpoints[0].x, midpoints[0].y)
    for p, m in zip(points[1:], midpoints[1:]):
        canvas.quadratic_curve_to(p.x, p.y, m.x, m.y)
    canvas.line_to(points[-1].x, points[-1].y)
    canvas.stroke()

In [ ]:
button = w.Button(description="Clear", button_style="danger")
canvas = Canvas(width=640, height=480, style=dict(border="1px solid magenta"))
canvas.stroke_style = pick_color("blue")
canvas.line_width = 4
canvas.line_cap = "round"


def clear():
    canvas.clear()


button.on_click(lambda b: clear())

In [ ]:
display(w.HBox([canvas, button]))

In [ ]:
p1 = XY(100, 200)
p2 = XY(500, 400)
points = tuple(split_line(p1, p2, n=16))
points = tuple(jig_line(points, 25))

for p in points:
    canvas.fill_style = "#000000"
    canvas.fill_circle(p.x, p.y, 6)

In [ ]:
# canvas.stroke_style = pick_color("blue")
# stroke_straight_path(canvas, points)

In [ ]:
canvas.stroke_style = pick_color("orange")
stroke_quadsmooth_path(canvas, points)